##Reading from Bronze

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window


In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

##Transformation

In [0]:
display(df)

columns = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
}

In [0]:
dup_prd_id_df = df.groupBy("prd_id") \
                .count() \
                .filter(col("count") > 1)
display(dup_prd_id_df)

dup_prd_key_df = df.groupBy("prd_key") \
    .count() \
    .filter(F.col("count") > 1)
display(dup_prd_key_df)

## product key duplicates are from historical data

In [0]:

columns_map = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_col, new_col in columns_map.items():
    df = df.withColumnRenamed(old_col, new_col)

display(df)

In [0]:
for field in df.schema.fields:
    if isinstance(field, StringType):
        df.withColumn(field.name, trim(col(field.name)))

In [0]:
df = df.withColumn("category_id", F.regexp_replace(F.substring(col("product_key"), 1, 5), "-", "_"))
df = df.withColumn("product_key", F.substring(col("product_key"), 7, F.length(col("product_key"))))

display(df)

In [0]:
df = (
    df
    .withColumn(
        "product_line",
        F.when(F.upper(trim(col("product_line"))) == "M", "Mountain")
         .when(F.upper(trim(col("product_line"))) == "R", "Road")
         .when(F.upper(trim(col("product_line"))) == "S", "Other Sales")
         .when(F.upper(trim(col("product_line"))) == "T", "Touring")
         .otherwise("n/a")
    )
)

In [0]:
df = df.withColumn("product_cost", F.coalesce(col("product_cost"), F.lit(0)))
df = df.withColumn("start_date", col("start_date").cast(DateType()))
df = df.withColumn("end_date", col("end_date").cast(DateType()))

display(df)

## Writing to silver

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10